# ProWrite AI — Préparation du corpus Enron
Notebook Colab : lire `emails.csv` par blocs, extraire le texte, filtrer, dédupliquer, produire un fichier compressé et un échantillon de contrôle.

**Important :** les e-mails seuls ne sont pas des paires `input → target`. La génération de messages professionnels exigera des paires supplémentaires. Le masquage ci-dessous n'est pas une anonymisation garantie.

In [ ]:
import os, re, csv, gzip, random, hashlib, html
from email import policy
from email.parser import Parser
import pandas as pd
from tqdm.auto import tqdm

# Si le fichier est dans Google Drive, monter Drive puis modifier INPUT_FILE.
INPUT_FILE = '/content/emails.csv'
OUTPUT_DIR = '/content/prowrite_output'
MAX_EMAILS = 200_000  # augmenter seulement après inspection
CHUNK_SIZE = 5_000
SAMPLE_SIZE = 5_000
MIN_WORDS, MAX_WORDS = 10, 120
os.makedirs(OUTPUT_DIR, exist_ok=True)
CLEAN_FILE = os.path.join(OUTPUT_DIR, 'enron_clean.csv.gz')
SAMPLE_FILE = os.path.join(OUTPUT_DIR, 'enron_sample_5000.csv')
assert os.path.isfile(INPUT_FILE), f'Fichier introuvable : {INPUT_FILE}'
print('Fichier trouvé :', round(os.path.getsize(INPUT_FILE)/1024**2, 1), 'Mo')
print('Colonnes :', pd.read_csv(INPUT_FILE, nrows=1).columns.tolist())

In [ ]:
parser = Parser(policy=policy.default)
def extract_body(raw):
    if not isinstance(raw, str) or not raw.strip():
        return None
    try:
        msg = parser.parsestr(raw)
        part = msg.get_body(preferencelist=('plain',))
        body = part.get_content() if part is not None else msg.get_payload()
        if not isinstance(body, str):
            return None
        # Tronquer les chaînes de réponses/transferts au lieu de garder d'autres messages.
        body = re.split(r'(?im)^\s*(?:-{3,}\s*(?:original message|forwarded by)|from:|sent:|on .{10,}wrote:)', body, maxsplit=1)[0]
        body = re.split(r'(?im)^\s*_{8,}\s*$', body, maxsplit=1)[0]
        body = re.sub(r'https?://\S+|www\.\S+', '[URL]', body)
        body = re.sub(r'[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}', '[EMAIL]', body)
        body = re.sub(r'(?<!\w)(?:\+?\d[\d(). -]{7,}\d)(?!\w)', '[PHONE]', body)
        body = html.unescape(body)
        body = re.sub(r'\s+', ' ', body).strip()
        words = body.split()
        if not MIN_WORDS <= len(words) <= MAX_WORDS:
            return None
        if len(re.findall(r'[A-Za-z]', body)) < 20:
            return None
        return body
    except Exception:
        return None

print('Nettoyage prêt.')

In [ ]:
seen_hashes = set()
sample_rows = []
rng = random.Random(42)
processed = kept = rejected = duplicates = 0
# Cette cellule recommence proprement le résultat si elle est relancée.
with gzip.open(CLEAN_FILE, 'wt', encoding='utf-8', newline='') as fout:
    writer = csv.writer(fout)
    writer.writerow(['text'])
    reader = pd.read_csv(INPUT_FILE, usecols=['message'], chunksize=CHUNK_SIZE, dtype={'message':'string'}, on_bad_lines='skip')
    for chunk in tqdm(reader, desc='Nettoyage'):
        for raw in chunk['message']:
            processed += 1
            text = extract_body(raw)
            if text is None:
                rejected += 1
                continue
            fingerprint = hashlib.sha256(text.lower().encode('utf-8')).digest()
            if fingerprint in seen_hashes:
                duplicates += 1
                continue
            seen_hashes.add(fingerprint)
            writer.writerow([text])
            kept += 1
            if len(sample_rows) < SAMPLE_SIZE:
                sample_rows.append(text)
            else:
                j = rng.randrange(kept)
                if j < SAMPLE_SIZE:
                    sample_rows[j] = text
            if kept >= MAX_EMAILS:
                break
        if kept >= MAX_EMAILS:
            break

pd.DataFrame({'text': sample_rows}).to_csv(SAMPLE_FILE, index=False)
print({'records_processed':processed, 'kept':kept, 'rejected':rejected, 'duplicates':duplicates})
print('Sorties :', CLEAN_FILE, SAMPLE_FILE, sep='\n')

In [ ]:
sample = pd.read_csv(SAMPLE_FILE)
print('Échantillon :', sample.shape)
print('Doublons :', sample.duplicated('text').sum())
print('Longueur (mots) :')
display(sample.text.str.split().str.len().describe())
# Montrer des messages courts pour vérification manuelle.
display(sample.head(10))
print('ATTENTION : inspecter et anonymiser avant de partager le fichier.')

## Récupérer les fichiers
Dans Colab, ouvre l'onglet **Files** puis le dossier `/content/prowrite_output` pour télécharger l'échantillon. Pour conserver le grand fichier après la fermeture de la session, copie `enron_clean.csv.gz` dans Google Drive.

**Suite du projet :** garder CoEdIT séparé, construire des paires de réécriture professionnelle, puis diviser train/validation/test avant l'entraînement de l'Encoder–Decoder.